In [28]:
# Importing libraries
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, regularizers, callbacks
import tensorflow as tf

In [29]:
# Loading and cleaning data from cicddos2019 dataset
df = pd.read_csv('/kaggle/input/cic-ddos2019-ml/output.csv', low_memory=False)
print(f"Raw data shape: {df.shape}")

Raw data shape: (125170, 78)


In [30]:
# Removing columns with >50% missing cells
nan_thresh = len(df) * 0.5
df = df.dropna(axis=1, thresh=nan_thresh)

# Removing rows with >5% missing cells
row_nan_thresh = int(df.shape[1] * 0.95)
df = df.dropna(thresh=row_nan_thresh)

# Removing columns with all zeros or all negative (except 'Label')
for col in df.select_dtypes(include=[np.number]).columns:
    if col != 'Label':
        if (df[col] == 0).all() or (df[col] < 0).all():
            df = df.drop(col, axis=1)

# Removing duplicates
df = df.drop_duplicates()

In [31]:
# Imputation and Encoding
num_cols = df.select_dtypes(include=[np.number]).columns.drop('Label', errors='ignore')
cat_cols = df.select_dtypes(exclude=[np.number]).columns.drop('Label', errors='ignore')

if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [32]:
# Encoding categorical columns
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Encoding Labels if not already numeric
if df['Label'].dtype == 'object':
    df['Label'] = LabelEncoder().fit_transform(df['Label'])

In [33]:
# Feature Scaling
scaler = MinMaxScaler()
feature_cols = [c for c in df.columns if c != 'Label']
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# Splitting data into Train/Test sets
X = df.drop('Label', axis=1).values
y = df['Label'].values

In [34]:
# Saving the original string labels for the encoder
le = LabelEncoder()
le.fit(df['Label'].astype(str))  # This captures the mapping from string to number

# encoding the labels for modeling
df['Label'] = le.transform(df['Label'].astype(str))

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

num_classes = len(np.unique(y_train))
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

Train shape: (97340, 65), Test shape: (24336, 65)


In [36]:
# SSDAE (Stacked Sparse Denoising Autoencoder)
def build_ssdae(input_dim, encoding_dim=64):
    input_layer = layers.Input(shape=(input_dim,))
    noise = layers.GaussianNoise(0.1)(input_layer)
    encoded = layers.Dense(128, activation='relu', activity_regularizer=regularizers.l1(1e-5))(noise)
    encoded = layers.Dense(encoding_dim, activation='relu')(encoded)
    decoded = layers.Dense(128, activation='relu')(encoded)
    decoded = layers.Dense(input_dim, activation='linear')(decoded)
    autoencoder = models.Model(input_layer, decoded)
    encoder = models.Model(input_layer, encoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder, encoder

print("\n=== SSDAE PRETRAINING ===")
input_dim = X_train.shape[1]
ssdae, ssdae_encoder = build_ssdae(input_dim, encoding_dim=64)
ssdae.fit(X_train, X_train, epochs=5, batch_size=1024, verbose=1, validation_split=0.1)


=== SSDAE PRETRAINING ===
Epoch 1/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.1053 - val_loss: 0.0144
Epoch 2/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0134 - val_loss: 0.0069
Epoch 3/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0070 - val_loss: 0.0046
Epoch 4/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0050 - val_loss: 0.0038
Epoch 5/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0042 - val_loss: 0.0033


In [37]:
# Transforming features using SSDAE encoder
X_train_encoded = ssdae_encoder.predict(X_train)
X_test_encoded = ssdae_encoder.predict(X_test)

3042/3042 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
761/761 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [38]:
# CNN-LSTM Model
def build_hybrid_cnn_lstm(input_dim, num_classes):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Reshape((input_dim, 1))(inp)
    # CNN layers
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(32, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    # LSTM layer
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inp, out)
    return model

In [39]:
model = build_hybrid_cnn_lstm(X_train_encoded.shape[1], num_classes)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape_1 (Reshape)                  │ (None, 64, 1)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 64, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_2 (MaxPooling1D)       │ (None, 32, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_3 (Conv1D)                    │ (None, 32, 32)              │           6,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_3 (MaxPooling1D)       │ (None, 16, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          24,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 128)                 │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 8)                   │           1,032 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 40,616 (158.66 KB)

 Trainable params: 40,616 (158.66 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
cb = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    callbacks.ModelCheckpoint('best_model.keras', save_best_only=True),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
    callbacks.TensorBoard(log_dir='./logs')
]

In [41]:
# Training the model
history = model.fit(
    X_train_encoded, y_train,
    epochs=50,
    batch_size=1024,
    validation_split=0.15,
    class_weight=class_weight_dict,
    callbacks=cb,
    verbose=1
)

Epoch 1/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.0497 - loss: 2.0864 - val_accuracy: 0.4139 - val_loss: 1.8320 - learning_rate: 0.0010
Epoch 2/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2811 - loss: 1.7026 - val_accuracy: 0.4309 - val_loss: 1.5703 - learning_rate: 0.0010
Epoch 3/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4832 - loss: 1.5041 - val_accuracy: 0.5971 - val_loss: 1.1112 - learning_rate: 0.0010
Epoch 4/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6592 - loss: 1.2442 - val_accuracy: 0.5178 - val_loss: 1.2346 - learning_rate: 0.0010
Epoch 5/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6852 - loss: 1.2234 - val_accuracy: 0.8411 - val_loss: 0.6579 - learning_rate: 0.0010
Epoch 6/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7924 - loss: 0.9861 - val_accuracy: 0.8298 - val_loss: 0.6714 - learning_rate: 0.0010
Epoch 7/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7798 - loss: 1.0109 - val_acc

In [42]:
#Evaluating the model
print("\n=== EVALUATION ===")
model.load_weights('best_model.keras')
loss, acc = model.evaluate(X_test_encoded, y_test)
print(f"\nFinal Test Accuracy: {acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_encoded)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred_classes))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_classes))


=== EVALUATION ===
761/761 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9217 - loss: 0.3874

Final Test Accuracy: 0.9218
761/761 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.90      0.94      9020
           1       0.82      0.97      0.89       377
           2       0.91      0.77      0.83      1687
           3       0.23      0.84      0.36        95
           4       0.11      0.01      0.01       137
           5       0.92      0.99      0.96      9450
           6       0.98      0.90      0.93      3559
           7       0.01      0.64      0.03        11

    accuracy                           0.92     24336
   macro avg       0.62      0.75      0.62     24336
weighted avg       0.95      0.92      0.93     24336


Confusion Matrix:
[[8101    0   27   56    7  791   25   13]
 [   1  366   10    0    0    0    0    0]
 [   0   83 1293    0    0    1   32  278]
 [   4    0  

In [43]:
# Pick 5 random indices from the test set for demo
np.random.seed(42)
demo_indices = np.random.choice(X_test_encoded.shape[0], 5, replace=False)

# Selecting the 5 samples
X_demo = X_test_encoded[demo_indices]
y_true = y_test[demo_indices]

# Getting model predictions
y_pred_probs = model.predict(X_demo)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

try:
    # If le.classes_ contains string names, use it
    true_label_names = [le.classes_[num] for num in y_true]
    pred_label_names = [le.classes_[num] for num in y_pred_classes]
    class_mapping = {i: name for i, name in enumerate(le.classes_)}
except Exception:
    # If le is not available or classes_ are numbers, use a manual mapping
    # Replace the mapping below with your actual class names if needed!
    class_mapping = {
        0: 'BENIGN',
        1: 'DDoS-RSTFINFlood',
        2: 'DDoS-UDPFlood',
        3: 'DDoS-SYNFlood',
        4: 'DDoS-PSHACK_Flood',
        5: 'DDoS-ACK_FragmentFlood',
        6: 'DDoS-UDP_FragmentFlood',
        7: 'DDoS-TCP_FragmentFlood'
    }
    true_label_names = [class_mapping.get(num, str(num)) for num in y_true]
    pred_label_names = [class_mapping.get(num, str(num)) for num in y_pred_classes]

try:
    input_features = pd.DataFrame(X_test[demo_indices], columns=feature_cols)
except Exception:
    input_features = pd.DataFrame(X_test[demo_indices])

results_df = pd.DataFrame({
    'True Label (num)': y_true,
    'Predicted Label (num)': y_pred_classes,
    'Model Confidence': y_pred_probs.max(axis=1)
})

print("Demo: 5 Random Test Samples")
print(results_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Demo: 5 Random Test Samples
   True Label (num)  Predicted Label (num)  Model Confidence
0                 2                      2          0.599785
1                 0                      5          0.532789
2                 5                      5          0.929683
3                 0                      0          0.800150
4                 6                      6          0.856900
